# BTC 4-Hourly Technical Indicators

Extended version with comprehensive technical indicators tailored for the 4-hour (4H) timeframe.

**Categories**: Order Flow · Returns · Momentum · Trend · Volatility · Volume · Multi-Timeframe · Temporal

**Note**: Dec 2017 is downloaded as warmup to avoid NaN at the 2018-01-01 boundary (longest lookback = SMA-200 = 200 candles; Dec 2017 provides ~186 4H candles, which is enough to warm up most indicators, and earlier 2017 data isn't strictly necessary).

In [1]:
import pandas as pd
import numpy as np
import requests
import zipfile
import io

# --- 1. CONFIGURATION ---
symbol = 'BTCUSDT'
interval = '4h' # <-- UPDATED: Changed from '1h' to '4h'
base_url = "https://data.binance.vision/data/spot/monthly/klines"
all_frames = []
EPS = 1e-10  # Epsilon to prevent division-by-zero

print(f"📡 Downloading {symbol} 4-Hourly Data (Dec 2017 - Dec 2025)...") # <-- UPDATED

# --- 2. DOWNLOAD LOOP ---
# Dec 2017 is fetched to "warm up" rolling windows (SMA-200 needs 200 candles;
# Dec 2017 provides 186 4H candles, so almost all indicators are valid by Jan 1 2018). # <-- UPDATED
fetch_list = [(2017, 12)] + [(y, m) for y in range(2018, 2026) for m in range(1, 13)]

for year, month in fetch_list:
    m_str = f"{month:02d}"
    file_name = f"{symbol}-{interval}-{year}-{m_str}.zip"
    url = f"{base_url}/{symbol}/{interval}/{file_name}"
    
    try:
        response = requests.get(url, timeout=15)
        if response.status_code == 200:
            with zipfile.ZipFile(io.BytesIO(response.content)) as z:
                csv_name = file_name.replace('.zip', '.csv')
                with z.open(csv_name) as f:
                    all_frames.append(pd.read_csv(f, header=None))
            print(f"✅ Loaded {year}-{m_str}")
        else:
            print(f"⚠️ Skipped {year}-{m_str}: Status {response.status_code}")
    except requests.exceptions.RequestException as e:
        print(f"❌ Failed to download {year}-{m_str}: {e}")

📡 Downloading BTCUSDT 4-Hourly Data (Dec 2017 - Dec 2025)...
✅ Loaded 2017-12
✅ Loaded 2018-01
✅ Loaded 2018-02
✅ Loaded 2018-03
✅ Loaded 2018-04
✅ Loaded 2018-05
✅ Loaded 2018-06
✅ Loaded 2018-07
✅ Loaded 2018-08
✅ Loaded 2018-09
✅ Loaded 2018-10
✅ Loaded 2018-11
✅ Loaded 2018-12
✅ Loaded 2019-01
✅ Loaded 2019-02
✅ Loaded 2019-03
✅ Loaded 2019-04
✅ Loaded 2019-05
✅ Loaded 2019-06
✅ Loaded 2019-07
✅ Loaded 2019-08
✅ Loaded 2019-09
✅ Loaded 2019-10
✅ Loaded 2019-11
✅ Loaded 2019-12
✅ Loaded 2020-01
✅ Loaded 2020-02
✅ Loaded 2020-03
✅ Loaded 2020-04
✅ Loaded 2020-05
✅ Loaded 2020-06
✅ Loaded 2020-07
✅ Loaded 2020-08
✅ Loaded 2020-09
✅ Loaded 2020-10
✅ Loaded 2020-11
✅ Loaded 2020-12
✅ Loaded 2021-01
✅ Loaded 2021-02
✅ Loaded 2021-03
✅ Loaded 2021-04
✅ Loaded 2021-05
✅ Loaded 2021-06
✅ Loaded 2021-07
✅ Loaded 2021-08
✅ Loaded 2021-09
✅ Loaded 2021-10
✅ Loaded 2021-11
✅ Loaded 2021-12
✅ Loaded 2022-01
✅ Loaded 2022-02
✅ Loaded 2022-03
✅ Loaded 2022-04
✅ Loaded 2022-05
✅ Loaded 2022-06
✅ Lo

In [2]:
# --- 3. DYNAMIC TIMESTAMP CONVERSION ---
# Binance switched from millisecond to microsecond timestamps around mid-2024.
# This function handles both formats transparently.

if all_frames:
    df = pd.concat(all_frames, ignore_index=True)
    df.columns = ['open_time', 'open', 'high', 'low', 'close', 'volume',
                  'close_time', 'q_vol', 'count', 'tb_base', 'tb_quote', 'ignore']

    def convert_binance_time(ts):
        """Handle Binance's mid-2024 shift from millisecond to microsecond timestamps."""
        if ts > 10**14:
            return pd.to_datetime(ts, unit='us', utc=True)  # Microseconds
        return pd.to_datetime(ts, unit='ms', utc=True)      # Milliseconds

    print("⏳ Converting timestamps to strictly formatted UTC...")
    df['timestamp'] = df['open_time'].apply(convert_binance_time)

    df = df.dropna(subset=['timestamp']).sort_values('timestamp').reset_index(drop=True)
    df = df[['timestamp', 'open', 'high', 'low', 'close', 'volume',
             'q_vol', 'count', 'tb_base']].copy()

    print(f"✅ Timestamp conversion complete. Total raw rows: {len(df)}")
else:
    raise RuntimeError("No data was downloaded. Check your network connection.")

⏳ Converting timestamps to strictly formatted UTC...
✅ Timestamp conversion complete. Total raw rows: 17702


### 3.5 Data Continuity & Handling Missing Observations

In high-frequency cryptocurrency datasets, missing 4-hour observations frequently occur due to exchange server maintenance (e.g., Binance's February 2018 outage) or API downtime. To ensure mathematical integrity for rolling technical indicators and prevent look-ahead bias, missing data is handled using strict time-series imputation rules:

1. **State Variables (Open, High, Low, Close):** Handled utilizing Last Observation Carried Forward (LOCF). When an exchange is offline, the theoretical market value of the asset remains at the last agreed-upon traded price until trading resumes. Interpolation is strictly avoided to prevent future data leakage.
2. **Event Variables (Volume, Trade Count):** Filled with strictly `0`. During a server outage, no physical transactions occur. Forward-filling volume would falsely inject synthetic, high-volume trading activity into blackout periods, severely distorting volume-weighted metrics like VWAP and OBV.

In [3]:
# --- 3.5 DATA CONTINUITY & GAP FILLING ---
print("🔍 Checking for missing 4-hour periods in raw OHLCV data...")

# 1. Define the perfect timeline
min_ts = df['timestamp'].min()
max_ts = df['timestamp'].max()

# <-- UPDATED: Changed freq='h' to freq='4h'
perfect_timeline = pd.date_range(start=min_ts, end=max_ts, freq='4h') 

# 2. Identify missing periods
actual_timestamps = pd.DatetimeIndex(df['timestamp'])
missing_periods = perfect_timeline.difference(actual_timestamps)

print(f"Total Expected 4H Candles: {len(perfect_timeline)}")
print(f"Total Actual 4H Candles: {len(df)}")
print(f"Missing 4H Candles Found: {len(missing_periods)}")

if len(missing_periods) > 0:
    print(f"\n🩹 Patching {len(missing_periods)} missing periods using LOCF and Zero-Filling...")
    
    # 3. Align dataframe to the perfect timeline
    df = df.set_index('timestamp')
    df = df.reindex(perfect_timeline)
    df.index.name = 'timestamp'
    df = df.reset_index()
    
    # 4. Apply the academic filling strategy
    price_cols = ['open', 'high', 'low', 'close']
    vol_cols = ['volume', 'q_vol', 'count', 'tb_base']
    
    # LOCF for prices
    df[price_cols] = df[price_cols].ffill()
    # Zero-fill for volume/activity
    df[vol_cols] = df[vol_cols].fillna(0)
    
    # Safety net: Backfill prices ONLY if the very first candle(s) of 2017 are missing
    df[price_cols] = df[price_cols].bfill()
    
    print("✅ Gap filling complete. Dataset is now 100% continuous.")
else:
    print("✅ No missing periods found. Dataset is continuous.")

🔍 Checking for missing 4-hour periods in raw OHLCV data...
Total Expected 4H Candles: 17718
Total Actual 4H Candles: 17702
Missing 4H Candles Found: 16

🩹 Patching 16 missing periods using LOCF and Zero-Filling...
✅ Gap filling complete. Dataset is now 100% continuous.


## Feature Engineering — Technical Indicators

All features are computed on the full dataset (including Dec 2017 warmup).
NaN rows from rolling window warm-up are dropped later, before filtering to 2018+.

In [4]:
# ═══════════════════════════════════════════════════════════════
# 4.1 ORDER FLOW FEATURES (6 features)
# ═══════════════════════════════════════════════════════════════
print("📊 Computing Order Flow features...")

df['taker_sell_vol']       = df['volume'] - df['tb_base']
df['volume_delta']         = df['tb_base'] - df['taker_sell_vol']
df['avg_trade_size']       = np.where(df['count'] > 0, df['volume'] / df['count'], 0)
df['vol_delta_sma_24']     = df['volume_delta'].rolling(window=24).mean()
df['avg_trade_size_sma_24']= df['avg_trade_size'].rolling(window=24).mean()
df['vol_delta_ema_24']     = df['volume_delta'].ewm(span=24, adjust=False).mean()

print("   ✅ Order Flow: taker_sell_vol, volume_delta, avg_trade_size, vol_delta_sma_24, avg_trade_size_sma_24, vol_delta_ema_24")

📊 Computing Order Flow features...
   ✅ Order Flow: taker_sell_vol, volume_delta, avg_trade_size, vol_delta_sma_24, avg_trade_size_sma_24, vol_delta_ema_24


In [5]:
# ═══════════════════════════════════════════════════════════════
# 4.2 RETURNS & LOG RETURNS (8 features)
# ═══════════════════════════════════════════════════════════════
print("📊 Computing Returns...")

# <-- UPDATED: Shift values mathematically adjusted for 4H candles
# 1 row = 4 hours. 
# 4h = shift 1, 12h = shift 3, 24h = shift 6, 72h = shift 18
for shift, label in [(1, '4h'), (3, '12h'), (6, '24h'), (18, '72h')]:
    prev = df['close'].shift(shift)
    df[f'return_{label}']     = (df['close'] - prev) / (prev + EPS)
    df[f'log_return_{label}'] = np.log(df['close'] / (prev + EPS))

print("   ✅ Returns: 4h, 12h, 24h, 72h (Standard and Log)")

📊 Computing Returns...
   ✅ Returns: 4h, 12h, 24h, 72h (Standard and Log)


In [6]:
# ═══════════════════════════════════════════════════════════════
# 4.3 MOMENTUM INDICATORS (10 features)
# ═══════════════════════════════════════════════════════════════
print("📊 Computing Momentum indicators...")

# --- RSI (Wilder's method, 3 periods) ---
delta = df['close'].diff()
gain = (delta.where(delta > 0, 0)).fillna(0)
loss = (-delta.where(delta < 0, 0)).fillna(0)

for period in [7, 14, 28]:
    avg_gain = gain.ewm(com=period-1, adjust=False).mean()
    avg_loss = loss.ewm(com=period-1, adjust=False).mean()
    rs = avg_gain / (avg_loss + EPS)
    df[f'RSI_{period}'] = 100 - (100 / (1 + rs))

# --- Stochastic Oscillator (14-period %K, 3-period %D) ---
lowest_14  = df['low'].rolling(14).min()
highest_14 = df['high'].rolling(14).max()
df['stoch_k'] = 100 * (df['close'] - lowest_14) / (highest_14 - lowest_14 + EPS)
df['stoch_d'] = df['stoch_k'].rolling(3).mean()

# --- Williams %R (14-period) ---
df['williams_r'] = -100 * (highest_14 - df['close']) / (highest_14 - lowest_14 + EPS)

# --- Rate of Change (12 & 24 period) ---
df['roc_12'] = (df['close'] - df['close'].shift(12)) / (df['close'].shift(12) + EPS) * 100
df['roc_24'] = (df['close'] - df['close'].shift(24)) / (df['close'].shift(24) + EPS) * 100

# --- Commodity Channel Index (20-period) ---
tp = (df['high'] + df['low'] + df['close']) / 3
tp_sma = tp.rolling(20).mean()
tp_mad = tp.rolling(20).apply(lambda x: np.abs(x - x.mean()).mean(), raw=True)
df['cci_20'] = (tp - tp_sma) / (0.015 * tp_mad + EPS)

# --- Rolling Autocorrelation (Momentum Persistence) ---
# <-- UPDATED: Changed from return_1h.rolling(24) to return_4h.rolling(6)
df['autocorr_4h_24h'] = df['return_4h'].rolling(6).apply(lambda x: x.autocorr(lag=1) if len(x) > 1 else np.nan)

print("   ✅ Momentum: RSI_7, RSI_14, RSI_28, stoch_k, stoch_d, williams_r, roc_12, roc_24, cci_20, autocorr_4h_24h")

📊 Computing Momentum indicators...
   ✅ Momentum: RSI_7, RSI_14, RSI_28, stoch_k, stoch_d, williams_r, roc_12, roc_24, cci_20, autocorr_4h_24h


In [7]:
# ═══════════════════════════════════════════════════════════════
# 4.4 TREND INDICATORS (10 features)
# ═══════════════════════════════════════════════════════════════
print("📊 Computing Trend indicators...")

# EMAs
df['ema_12'] = df['close'].ewm(span=12, adjust=False).mean()
df['ema_26'] = df['close'].ewm(span=26, adjust=False).mean()

# SMAs
df['sma_50']  = df['close'].rolling(50).mean()
df['sma_200'] = df['close'].rolling(200).mean()

# MACD
df['MACD_line']      = df['ema_12'] - df['ema_26']
df['MACD_signal']    = df['MACD_line'].ewm(span=9, adjust=False).mean()
df['MACD_histogram'] = df['MACD_line'] - df['MACD_signal']

# Crossover signals (binary)
df['ema_12_26_cross']     = (df['ema_12'] > df['ema_26']).astype(int)
df['price_above_sma_200'] = (df['close'] > df['sma_200']).astype(int)

# --- Simple ADX Approximation (Directional Index) ---
# Note: Requires ATR_14, so we compute a localized True Range and ATR just for ADX 
# to keep this block independent, or rely on the later block. 
# Best practice is to calculate the specific ATR needed for ADX here:
up_move = df['high'] - df['high'].shift(1)
down_move = df['low'].shift(1) - df['low']
plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0)
minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0)

# Calculate localized TR for ADX to avoid dependency on 4.5 running first
tr_adx = pd.concat([
    df['high'] - df['low'],
    np.abs(df['high'] - df['close'].shift(1)),
    np.abs(df['low'] - df['close'].shift(1))
], axis=1).max(axis=1)
atr_14_adx = tr_adx.rolling(window=14).mean()

plus_di = 100 * pd.Series(plus_dm).ewm(alpha=1/14, adjust=False).mean() / (atr_14_adx + EPS)
minus_di = 100 * pd.Series(minus_dm).ewm(alpha=1/14, adjust=False).mean() / (atr_14_adx + EPS)
dx = 100 * np.abs(plus_di - minus_di) / (plus_di + minus_di + EPS)
df['adx_14'] = dx.ewm(alpha=1/14, adjust=False).mean()

print("   ✅ Trend: ema_12, ema_26, sma_50, sma_200, MACD_line, MACD_signal, MACD_histogram, ema_12_26_cross, price_above_sma_200, adx_14")

📊 Computing Trend indicators...
   ✅ Trend: ema_12, ema_26, sma_50, sma_200, MACD_line, MACD_signal, MACD_histogram, ema_12_26_cross, price_above_sma_200, adx_14


In [8]:
# ═══════════════════════════════════════════════════════════════
# 4.5 VOLATILITY INDICATORS (9 features)
# ═══════════════════════════════════════════════════════════════
print("📊 Computing Volatility indicators...")

# True Range
high_low   = df['high'] - df['low']
high_close = np.abs(df['high'] - df['close'].shift(1))
low_close  = np.abs(df['low'] - df['close'].shift(1))
tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)

# ATR
df['ATR_14'] = tr.rolling(window=14).mean()
df['ATR_7']  = tr.rolling(window=7).mean()

# --- Volatility Regime (ATR ratio) ---
df['atr_ratio_7_14'] = df['ATR_7'] / (df['ATR_14'] + EPS)

# Bollinger Bands (20-period, 2 sigma)
sma_20 = df['close'].rolling(20).mean()
std_20 = df['close'].rolling(20).std()
df['bb_upper'] = sma_20 + 2 * std_20
df['bb_lower'] = sma_20 - 2 * std_20
df['bb_mid']   = sma_20
df['bb_width'] = (df['bb_upper'] - df['bb_lower']) / (df['bb_mid'] + EPS)
df['bb_pct']   = (df['close'] - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'] + EPS)

# Realized Volatility (rolling std of log returns, 24h)
log_ret = np.log(df['close'] / (df['close'].shift(1) + EPS))
df['realized_vol_24'] = log_ret.rolling(24).std()

print("   ✅ Volatility: ATR_14, ATR_7, atr_ratio_7_14, bb_upper, bb_lower, bb_mid, bb_width, bb_pct, realized_vol_24")

📊 Computing Volatility indicators...
   ✅ Volatility: ATR_14, ATR_7, atr_ratio_7_14, bb_upper, bb_lower, bb_mid, bb_width, bb_pct, realized_vol_24


In [9]:
# ═══════════════════════════════════════════════════════════════
# 4.6 VOLUME INDICATORS (6 features)
# ═══════════════════════════════════════════════════════════════
print("📊 Computing Volume indicators...")

# Volume SMA & relative ratio
df['volume_sma_24'] = df['volume'].rolling(24).mean()
df['volume_ratio']  = df['volume'] / (df['volume_sma_24'] + EPS)

# (volume_delta, vol_delta_sma_24, vol_delta_ema_24 already computed in Order Flow)

# On-Balance Volume (OBV) & its EMA
direction = np.sign(df['close'].diff()).fillna(0)
df['obv']        = (df['volume'] * direction).cumsum()
df['obv_ema_24'] = df['obv'].ewm(span=24, adjust=False).mean()

# --- OBV Slope (Volume confirming price) ---
df['obv_slope_5'] = df['obv'].diff(5) / 5.0

# VWAP ratio (close relative to volume-weighted avg price)
vwap = df['q_vol'] / (df['volume'] + EPS)
df['vwap_ratio'] = df['close'] / (vwap + EPS)

print("   ✅ Volume: volume_sma_24, volume_ratio, obv, obv_ema_24, obv_slope_5, vwap_ratio")

📊 Computing Volume indicators...
   ✅ Volume: volume_sma_24, volume_ratio, obv, obv_ema_24, obv_slope_5, vwap_ratio


In [10]:
# ═══════════════════════════════════════════════════════════════
# 4.7 MULTI-TIMEFRAME FEATURES (5 features)
# ═══════════════════════════════════════════════════════════════
print("📊 Computing Multi-Timeframe features...")

# <-- UPDATED: Windows adjusted for 4H base timeframe
# close_4h_sma is redundant on a 4H chart, so we scale these up to 24h (6 periods)
df['close_24h_sma']  = df['close'].rolling(6).mean()  # 6 * 4h = 24h
df['volume_24h_sum'] = df['volume'].rolling(6).sum()  # 6 * 4h = 24h

# 12h high/low requires a rolling window of 3 periods (3 * 4h = 12h)
df['high_12h']      = df['high'].rolling(3).max()
df['low_12h']       = df['low'].rolling(3).min()
df['range_12h_pct'] = (df['high_12h'] - df['low_12h']) / (df['close'] + EPS)

print("   ✅ Multi-TF: close_24h_sma, volume_24h_sum, high_12h, low_12h, range_12h_pct")

📊 Computing Multi-Timeframe features...
   ✅ Multi-TF: close_24h_sma, volume_24h_sum, high_12h, low_12h, range_12h_pct


In [11]:
# ═══════════════════════════════════════════════════════════════
# 4.8 TEMPORAL & CYCLICAL FEATURES (4 features)
# ═══════════════════════════════════════════════════════════════
print("📊 Computing Temporal features...")

# Cyclical Time Features (Helps models learn daily/weekly seasonality)
df['hour_sin'] = np.sin(2 * np.pi * df['timestamp'].dt.hour / 24.0)
df['hour_cos'] = np.cos(2 * np.pi * df['timestamp'].dt.hour / 24.0)
df['day_sin'] = np.sin(2 * np.pi * df['timestamp'].dt.dayofweek / 7.0)
df['day_cos'] = np.cos(2 * np.pi * df['timestamp'].dt.dayofweek / 7.0)

print("   ✅ Temporal: hour_sin, hour_cos, day_sin, day_cos")

📊 Computing Temporal features...
   ✅ Temporal: hour_sin, hour_cos, day_sin, day_cos


In [12]:
# --- 5. FILTER & EXPORT ---
print("✂️ Filtering data to start exactly from 2018-01-01 00:00:00 UTC...")

# 1. Filter by date FIRST to remove the Dec 2017 warmup data
df = df[df['timestamp'] >= '2018-01-01'].reset_index(drop=True)

# 2. Any remaining NaNs are mathematical errors caused by the flat-line gap fills
# (e.g., Autocorrelation of a flat price = NaN). We safely fill these with 0.
df = df.fillna(0)

# Add Trend Label
#df['trend'] = np.where(df['close'] > df['open'], 'Bullish', 'Bearish')

# Export
# <-- UPDATED: Changed filename to reflect 4-hour data
df.to_csv('btc_4hourly_indicators.csv', index=False)

print(f"✨ Success! Total Rows: {len(df)}")
print(f"Data starts at: {df['timestamp'].min()}")
print(f"Data ends at: {df['timestamp'].max()}")
print(f"Total features: {len(df.columns)}")

✂️ Filtering data to start exactly from 2018-01-01 00:00:00 UTC...
✨ Success! Total Rows: 17532
Data starts at: 2018-01-01 00:00:00+00:00
Data ends at: 2025-12-31 20:00:00+00:00
Total features: 67


In [13]:
# --- 6. VERIFICATION ---
print("\n" + "=" * 60)
print("  VERIFICATION REPORT")
print("=" * 60)

"""
trend_counts = df['trend'].value_counts()
print(f"\nTrend Distribution:")
print(trend_counts)
"""
print(f"\nNaN Check: {df.isna().sum().sum()} remaining NaNs")

# Sanity checks
print(f"\nSanity Checks:")
for col in ['RSI_7', 'RSI_14', 'RSI_28']:
    print(f"  {col}: [{df[col].min():.2f}, {df[col].max():.2f}] (expected 0-100)")
for col in ['ATR_14', 'ATR_7']:
    print(f"  {col}: min={df[col].min():.4f} (expected >= 0)")
print(f"  bb_pct: [{df['bb_pct'].min():.4f}, {df['bb_pct'].max():.4f}]")

print(f"\n📊 Quick Stats:")
print(df.describe().round(4).to_string())


  VERIFICATION REPORT

NaN Check: 0 remaining NaNs

Sanity Checks:
  RSI_7: [1.24, 98.01] (expected 0-100)
  RSI_14: [5.37, 94.99] (expected 0-100)
  RSI_28: [14.44, 89.44] (expected 0-100)
  ATR_14: min=23.0457 (expected >= 0)
  ATR_7: min=16.3857 (expected >= 0)
  bb_pct: [-0.5037, 1.5471]

📊 Quick Stats:
              open         high          low        close       volume         q_vol         count      tb_base  taker_sell_vol  volume_delta  avg_trade_size  vol_delta_sma_24  avg_trade_size_sma_24  vol_delta_ema_24   return_4h  log_return_4h  return_12h  log_return_12h  return_24h  log_return_24h  return_72h  log_return_72h       RSI_7      RSI_14      RSI_28     stoch_k     stoch_d  williams_r      roc_12      roc_24      cci_20  autocorr_4h_24h       ema_12       ema_26       sma_50      sma_200   MACD_line  MACD_signal  MACD_histogram  ema_12_26_cross  price_above_sma_200      adx_14      ATR_14       ATR_7  atr_ratio_7_14     bb_upper     bb_lower       bb_mid    bb_width    

In [14]:
import pandas as pd

# 1. Read the output file
file_path = 'btc_4hourly_indicators.csv'  # Update this if your file name is different
print(f"📂 Loading file: {file_path}...")
df = pd.read_csv(file_path)

# Ensure the timestamp column is read as datetime objects
df['timestamp'] = pd.to_datetime(df['timestamp'])

# 2. Check for NaN values
total_nans = df.isna().sum().sum()
columns_with_nans = df.isna().sum()[df.isna().sum() > 0]

# 3. Calculate expected vs. actual periods
min_ts = df['timestamp'].min()
max_ts = df['timestamp'].max()

# Generate a perfect timeline
# Note: If checking an hourly file, change freq='4h' to freq='h'
perfect_timeline = pd.date_range(start=min_ts, end=max_ts, freq='4h')

expected_rows = len(perfect_timeline)
actual_rows = len(df)
missing_rows = expected_rows - actual_rows

# --- Print Diagnostic Report ---
print("\n" + "=" * 40)
print("  DATASET DIAGNOSTIC REPORT")
print("=" * 40)

print(f"\n⚠️ NaN Check:")
print(f"Total missing values (NaNs): {total_nans}")
if total_nans > 0:
    print("\nColumns containing NaNs:")
    print(columns_with_nans.to_string())

print(f"\n⏱️ Continuity Check:")
print(f"Start Time:    {min_ts}")
print(f"End Time:      {max_ts}")
print("-" * 40)
print(f"Expected Rows: {expected_rows} (Based on perfectly continuous time)")
print(f"Actual Rows:   {actual_rows}")
print(f"Missing Rows:  {missing_rows}")

if missing_rows == 0 and total_nans == 0:
    print("\n✅ Status: PERFECT. The dataset is clean and continuous.")
else:
    print("\n❌ Status: WARNING. The dataset has missing rows or NaN values.")

📂 Loading file: btc_4hourly_indicators.csv...

  DATASET DIAGNOSTIC REPORT

⚠️ NaN Check:
Total missing values (NaNs): 0

⏱️ Continuity Check:
Start Time:    2018-01-01 00:00:00+00:00
End Time:      2025-12-31 20:00:00+00:00
----------------------------------------
Expected Rows: 17532 (Based on perfectly continuous time)
Actual Rows:   17532
Missing Rows:  0

✅ Status: PERFECT. The dataset is clean and continuous.
